In [67]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [68]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stream": True,
    }

    if system:
        params["system"] = system

    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)

    # Collect streamed text (fallbacks handled by returning string form)
    text = ""
    try:
        for event in message:
            if getattr(event, "type", None) == "content_block_delta":
                delta = getattr(event, "delta", None)
                if delta and hasattr(delta, "text"):
                    text += delta.text
        if text:
            return text
    except TypeError:
        # message is not iterable (non-streaming response)
        pass

    # Fallback: try to extract from message.content
    try:
        content = getattr(message, "content", None)
        if content:
            # content may be a list of blocks
            text = ""
            for block in content:
                if isinstance(block, dict) and "text" in block:
                    text += block["text"]
                elif hasattr(block, "text"):
                    text += block.text
            if text:
                return text
    except Exception:
        pass

    return str(message)


In [69]:
import json


def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case['task']}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

# Implementing a Model Grader
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case['task']}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])

    # Ensure we strip surrounding markdown if present
    eval_text = eval_text.strip()
    if eval_text.startswith("```") and eval_text.endswith("```"):
        eval_text = eval_text.strip("`\n ")

    return json.loads(eval_text)

# Integrating Grading into Your Workflow
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade.get("score")
    reasoning = model_grade.get("reasoning")
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }




In [70]:
# Finally, calculate an average score across all test cases:
import json
from statistics import mean
results = []
def run_eval(dataset):
    
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

with open("dataset.json") as f:
    dataset = json.load(f)
results = run_eval(dataset)

print(json.dumps(results, indent=2))

Average score: 6.833333333333333
[
  {
    "output": "# AWS Account ID Extractor\n\nHere's a Python function that extracts the AWS account ID from an ARN string:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the 12-digit AWS account ID from an ARN string.\n    \n    ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Args:\n        arn (str): The AWS ARN string\n        \n    Returns:\n        str: The 12-digit AWS account ID\n        \n    Raises:\n        ValueError: If the ARN format is invalid or account ID is not found\n    \"\"\"\n    try:\n        # Split the ARN by colons\n        parts = arn.split(':')\n        \n        # ARN should have at least 6 parts\n        if len(parts) < 6:\n            raise ValueError(\"Invalid ARN format: insufficient parts\")\n        \n        # Account ID is the 5th element (index 4)\n        account_id = parts[4]\n        \n        # Validate that account ID is 12 